In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim


In [2]:
import torch
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F

# 1. 准备 MNIST 数据集（只取训练集）
transform = transforms.Compose([
    transforms.ToTensor(),                  # [0,255] -> [0.0,1.0], shape (1,28,28)
    transforms.Normalize((0.1307,), (0.3081,))
])
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
loader = DataLoader(mnist_train, batch_size=256, shuffle=False, num_workers=2)

# 2. 统计每个类别的样本总和与计数
num_classes = 10
# 累计向量和： shape (10, 784)
sum_vectors = torch.zeros((num_classes, 28*28), dtype=torch.float32)
# 每个类的样本数
counts = torch.zeros(num_classes, dtype=torch.long)

for imgs, labels in loader:
    # imgs: (B,1,28,28) -> 展平 (B,784)
    flat = imgs.view(-1, 28*28)
    for c in range(num_classes):
        mask = (labels == c)
        if mask.any():
            sum_vectors[c] += flat[mask].sum(dim=0)
            counts[c] += mask.sum()

# 3. 计算类均值向量
# 避免除以 0
mean_vectors = sum_vectors / counts.unsqueeze(1).float()   # shape (10, 784)

# 4. 计算余弦相似度矩阵
# 先将均值向量归一化
normed = F.normalize(mean_vectors, p=2, dim=1)             # 每行 L2 归一化
# 矩阵乘得到相似度 (10,10)
sim_matrix = torch.mm(normed, normed.t())

# 5. 打印结果
# 转为 numpy 更易阅读
import pandas as pd
df = pd.DataFrame(
    sim_matrix.cpu().numpy(),
    index=[str(i) for i in range(num_classes)],
    columns=[str(i) for i in range(num_classes)]
)
print("Digit Similarity Matrix (cosine between PyTorch mean-images):")
print(df.round(3))


ModuleNotFoundError: No module named 'pandas'

In [5]:
from scipy.io import loadmat

mat = loadmat('Mirflickr.mat')  # 路径根据你的数据文件
X = mat['data']  # shape: [M, d]
partial_y = mat['partial_target'].T  # shape: [M, Q]
true_y = mat['target'].T  # shape: [M, Q]
print(partial_y)

[[0 0 0 ... 1 0 0]
 [0 0 0 ... 0 0 0]
 [0 0 0 ... 0 0 0]
 ...
 [0 0 1 ... 0 0 1]
 [0 0 0 ... 1 1 1]
 [0 0 0 ... 1 1 1]]


In [7]:
import torch
s = torch.tensor([[1, 2], [2, 3]])
print(s)
print(s.sum(dim=1).shape)
print(s.sum(dim=1, keepdim=True).shape)

tensor([[1, 2],
        [2, 3]])
torch.Size([2])
torch.Size([2, 1])


In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(3, 6, 5)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(6, 16, 5)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 16 * 5 * 5)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)
        return x

net = Net()
print(net)

In [ ]:
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9)


In [ ]:
# Print model's state_dict
print("Model's state_dict:")
for param_tensor in net.state_dict():
    print(param_tensor, "\t", net.state_dict()[param_tensor].size())

print()

# Print optimizer's state_dict
print("Optimizer's state_dict:")
for var_name in optimizer.state_dict():
    print(var_name, "\t", optimizer.state_dict()[var_name])

In [ ]:
import torch
x = torch.arange(6)
print(x.shape, x.size())
x = x.reshape(2, 3)
print(x.shape, x.size())
print(torch.arange(6).numel())
print(x.reshape(3, -1))

In [ ]:
if 1 == 1.0:
    print(True)
else:
    print(False)

In [ ]:
x = torch.arange(12)
y = torch.arange(12)
print(id(x), id(y))
y[:] = x + y
print(id(y))
y =x + y
print(id(y))

In [ ]:
print(x > y, '\n', x == y, x < y)

In [ ]:
beta = 0.99
beta_zero = 0.9999
rou = 0.992
for  t in  range(400):
    beta_t = beta + (beta_zero - beta) * rou**(t)
    print(f'{beta_t:.10f}')

In [ ]:
r = torch.randperm(5)
print(r)

In [ ]:
from torchvision import datasets, transforms
train_dataset =  datasets.MNIST('./mnist_dataset', train=True, download=True)
print(len(train_dataset))

In [ ]:
from utils import plot
t = torch.arange(12).tolist()
plot(t)

In [ ]:
from torch.utils.tensorboard import SummaryWriter

writer = SummaryWriter('logs')
# writer.add_image()
for i in range(100):
    writer.add_scalar(tag='y=3*x', scalar_value=i, global_step=i)

writer.close() 

In [ ]:
A = [1, 2, 3]
for i in A:
    print(i)

In [ ]:
import random
train_dataset =  datasets.MNIST('./mnist_dataset', train=True, download=True)
l = (train_dataset.targets.numpy().tolist())
nums = []
for i in range(10000):
    nums.append(random.randint(0, 100))
nums.sort()

s = 0
for i in nums:
    for j in nums:
        s += abs(i - j)

sum_nums = sum(nums)
pre = 0
s_t = 0

for i, num in enumerate(nums):
    s_t += num * i - pre + (sum_nums - pre - num) - (len(nums) - i - 1) * num
    pre += num
print(s, s_t, s_t == s)

In [ ]:
print(chr(ord('a') + 25 - 1))

In [ ]:
import importlib
import utils
from utils import create_dirichlet_split_noniid
from torchvision import datasets, transforms
import torch
import numpy as np

importlib.reload(utils)

train_dataset =  datasets.MNIST('./mnist_dataset', train=True, download=True)
test_dataset = datasets.MNIST('./mnist_dataset', train=False, download=True)

train_data = train_dataset.data.to(torch.float)
train_labels = train_dataset.targets.numpy()

create_dirichlet_split_noniid(train_data=train_data,
                              train_labels=train_labels,
                              alpha=0.5,
                              clients_number=10)


print((np.argwhere(train_labels == 1)).flatten())

In [ ]:
from torch.distributions import Dirichlet
alpha = torch.tensor([0.1, 0.1])
dirichlet = Dirichlet(alpha)
print(dirichlet.sample())

In [ ]:
a = torch.arange(100)
c = 10

beta = 0.99
beta_zero = 0.9999
rou = 0.992

for t in range(1000):
    beta_t = beta + (beta_zero - beta) * rou**(t)
    y = 0
    for i in a:
        y += i
    y *= c * 2
    x = 0
    for j in a:
        for k in a:
            x += abs(j - k)
    print(x / y)

In [ ]:
import torch

a = torch.randn(4, 4)
print(a)
b = torch.tensor([[1, 2], [-1, 2]])
print(torch.argmax(a, 0))

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# 示例数据：真实标签和预测标签
y_true = [0, 1, 2, 2, 0, 1, 1, 2, 2, 0, 1, 2]
y_pred = [0, 0, 2, 1, 0, 2, 1, 0, 2, 0, 2, 2]

# 计算混淆矩阵
cm = confusion_matrix(y_true, y_pred)

# 绘制混淆矩阵
def plot_confusion_matrix(cm, classes, title='Confusion matrix', cmap=plt.cm.Blues):
    plt.imshow(cm, interpolation='nearest', cmap=cmap)
    plt.title(title)
    plt.colorbar()
    tick_marks = np.arange(cm.shape[0])
    plt.xticks(tick_marks, np.arange(cm.shape[0]), rotation=45)
    plt.yticks(tick_marks, np.arange(cm.shape[0]))

    fmt = 'd'
    thresh = cm.max() / 2.
    for i, j in np.ndindex(cm.shape):
        plt.text(j, i, format(cm[i, j], fmt),
                 horizontalalignment="center",
                 color="white" if cm[i, j] > thresh else "black")

    plt.ylabel('True label')
    plt.xlabel('Predicted label')
    plt.tight_layout()

# 类别名称
class_names = ['Class 0', 'Class 1', 'Class 2']

# 调用绘制函数
plot_confusion_matrix(cm, classes=class_names)
plt.figure


In [ ]:
import wandb
from utils import load_config

with wandb.init(project='config_example', config=load_config()) as run:
    print(run.config)

In [1]:
import tensorflow as tf

# 推荐方式：列出所有 GPU 设备
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print("检测到 GPU：")
    for gpu in gpus:
        print("  ", gpu)
else:
    print("未检测到 GPU。")

# 或者使用较旧的 tf.test 方法（可能会有警告）
print("tf.test.is_gpu_available():", tf.test.is_gpu_available())

未检测到 GPU。
Instructions for updating:
Use `tf.config.list_physical_devices('GPU')` instead.
tf.test.is_gpu_available(): False


In [1]:
import wandb

wandb.init(project="test", name="test_define_metric")
# define our custom x axis metric
wandb.define_metric("custom_step")
# define which metrics will be plotted against it
wandb.define_metric("validation_loss", step_metric="custom_step")

for i in range(10):
    log_dict = {
        "train_loss": 1 / (i + 1),
        "custom_step": i**2,
        "validation_loss": 1 / (i + 1),
    }
    wandb.log(log_dict)


wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: whitewall_9 (whitewall_9-jinan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [3]:
import wandb

wandb.init(project='test')
# define our custom x axis metric
wandb.define_metric("train/step")
# set all other train/ metrics to use this step
wandb.define_metric("train/*", step_metric="train/step")

for i in range(10):
    log_dict = {
        "train/step": 2**i,  # exponential growth w/ internal W&B step
        "train/loss": 1 / (i + 1),  # x-axis is train/step
        "train/accuracy": 1 - (1 / (1 + i)),  # x-axis is train/step
        "val/loss": 1 / (1 + i),  # x-axis is internal wandb step
    }
    wandb.log(log_dict)


In [3]:
from model import Mnist_CNN
import torch
import wandb
wandb.init(project='test')

model = Mnist_CNN()
torch.save(model.state_dict(), "model.pth")
artifact = wandb.Artifact("model", type="model")
artifact.add_file("model.pth")
wandb.log_artifact(artifact)
wandb.finish()

wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: whitewall_9 (whitewall_9-jinan-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [1]:
print(len("39e4a06e286b4613f18142047703373a63c91cff"))

40


In [5]:
import textwrap
name=f"""
        mnist_lr
        _long_tail_alpharandom_resample_random_clients
    """

print(textwrap.dedent(name))


mnist_lr
_long_tail_alpharandom_resample_random_clients



In [ ]:
import model
from model import simple_cnn

model = simple_cnn()
minst_tranform = transforms.Compose([
                transforms.Normalize((0.1307,), (0.3081,))
            ])
train_dataset =  datasets.MNIST('./fashion_mnist_dataset', train=True, download=True)
test_dataset = datasets.MNIST('./fashion_mnist_dataset', train=False, download=True)



模型总参数数量: 878538
simple_cnn(
  (conv1): Conv2d(3, 32, kernel_size=(5, 5), stride=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(5, 5), stride=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=1600, out_features=512, bias=True)
  (fc2): Linear(in_features=512, out_features=10, bias=True)
)
